# Inference Parameter Grid Search

This notebook automates a comprehensive grid search for YOLO inference hyperparameters (`conf`, `iou`, `imgsz`). The objective is to evaluate trade-offs between precision, recall, F1-score, mAP, and inference latency on the test set, ensuring optimal configuration for production deployment.

In [ ]:
import gc
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from itertools import product
from ultralytics import YOLO
import logging

# Suppress standard verbose logs to maintain clear metrics formatting
logging.getLogger('ultralytics').setLevel(logging.ERROR)
sns.set_theme(style='whitegrid')
print('Dependencies loaded successfully.')

## Configuration Setup
Define model paths, testing datasets, and specific parameters for grid exploration.

In [ ]:
class Config:
    # Using typical relative project structures
    MODEL_PATH = '../../training/models/YOLO26m_Batch4_March_Dataset/weights/best.pt'
    DATA_YAML = '../../configs/dataset.yaml'  # Ensure to route to correct YOLO test data yaml
    
    OUTPUT_CSV = 'inference_grid_search_results.csv'
    PLOT_OUTPUT = 'grid_search_visualization.png'
    
    # Fixed inference heuristics
    MAX_DET = 300
    HALF = False
    
    # Custom tracking metrics indexing
    MINORITY_CLASSES_INDICES = [1, 2, 3]  # Update to track specific lithology indices (e.g., Limestone, Sandstone)

config = Config()
print('Configuration initialized.')

## Grid Search Execution Module
Iterates comprehensively through hyperparameter combinations. Efficiently caches the model and saves intermediate progress iteratively.

In [ ]:
def run_grid_search(cfg):
    """
    Iterates across designated grid space parameters directly testing the inference latency, 
    robustness and boundary detection efficiency of a target weights file using YOLO framework natively.
    """
    # Phase 1: Coarse Grid Space Definition (Total: 60 iterations)
    param_grid = {
        'conf': [0.15, 0.20, 0.25, 0.30, 0.35, 0.40],
        'iou': [0.45, 0.55, 0.65, 0.70, 0.75],
        'imgsz': [640, 960]
    }
    
    keys = param_grid.keys()
    combinations = [dict(zip(keys, prod)) for prod in product(*(param_grid[k] for k in keys))]
    
    # Cache model loading for execution loop efficiency
    print(f'Loading model mapping: {cfg.MODEL_PATH}')
    model = YOLO(cfg.MODEL_PATH)
    
    results = []
    print(f'Initiating structural evaluation across {len(combinations)} parameter configurations...')
    
    for idx, params in enumerate(combinations, 1):
        print(f'\n[Iteration {idx}/{len(combinations)}] Config Mapping: {params}')
        
        try:
            # YOLO automated validation yields test set inference profiling natively based on standard datasets
            metrics = model.val(
                data=cfg.DATA_YAML,
                split='test',
                conf=params['conf'],
                iou=params['iou'],
                imgsz=params['imgsz'],
                max_det=cfg.MAX_DET,
                half=cfg.HALF,
                plots=False,
                save_json=False,
                verbose=False
            )
            
            # Aggregate standard matrix validations 
            precision = metrics.box.mp
            recall = metrics.box.mr
            # Macro F1 Representation 
            f1_score = 2 * (precision * recall) / (precision + recall + 1e-16)
            map50 = metrics.box.map50
            map50_95 = metrics.box.map
            
            # Hardware Latency Mapping (model speed translates runtime ms per unit frames)
            avg_latency_sec = metrics.speed['inference'] / 1000.0
            
            # Sub-level class inference (Minority distribution mapping fallback tracking)
            class_indices = metrics.box.ap_class_index
            minority_recall = recall
            if hasattr(metrics.box, 'r') and len(metrics.box.r) > 0:
                subset_r = [metrics.box.r[i] for i, cls_idx in enumerate(class_indices) if cls_idx in cfg.MINORITY_CLASSES_INDICES]
                if subset_r:
                    minority_recall = sum(subset_r) / len(subset_r)
            
            result_entry = {
                'conf': params['conf'],
                'iou': params['iou'],
                'imgsz': params['imgsz'],
                'precision': precision,
                'recall': recall,
                'f1_score': f1_score,
                'map50': map50,
                'map50_95': map50_95,
                'avg_latency_sec': avg_latency_sec,
                'minority_recall': minority_recall
            }
            
            results.append(result_entry)
            
            # Persist dynamically preventing crash data loss
            pd.DataFrame(results).to_csv(cfg.OUTPUT_CSV, index=False)
            gc.collect()
            
        except Exception as e:
            print(f'Execution validation failed evaluating {params}. Sequence trace error: {str(e)}')
            
    return pd.DataFrame(results)

## Visualization & Reporting Module
Render insights evaluating confidence thresholds, intersection dynamics, and computational execution variance.

In [ ]:
def plot_grid_search_insights(df, cfg):
    """
    Compiles analytical representations of validation outcomes mapping key deployment matrix constraints.
    """
    if df.empty:
        print('Insufficient processing volume mapping data plotting visuals.')
        return
        
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Precision-Recall Surface Contour
    ax = axes[0, 0]
    sns.scatterplot(data=df, x='recall', y='precision', hue='conf', style='iou', size='f1_score', palette='viridis', ax=ax)
    ax.set_title('Precision v Recall Trade-offs Matrix Profile')
    ax.set_xlabel('Global Aggregate Recall')
    ax.set_ylabel('Global Aggregate Precision')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # 2. Performance Linearity on NMS Intersections
    ax = axes[0, 1]
    for iou_val in df['iou'].unique():
        subset = df[df['iou'] == iou_val]
        ax.plot(subset['conf'], subset['f1_score'], marker='D', label=f'IoU NMS Depth: {iou_val}')
    ax.set_title('Macro F1 Evaluation Intersections (Across Thresholds)')
    ax.set_xlabel('Confidence Cutoff Threshold')
    ax.set_ylabel('Harmonic F1 Density')
    ax.legend()
    
    # 3. Output Processing Scale (Latency Index Mapping)
    ax = axes[1, 0]
    sns.boxplot(data=df, x='imgsz', y='avg_latency_sec', ax=ax, palette='mako')
    ax.set_title('Predictive Hardware Latency per Frame Standardizations')
    ax.set_xlabel('Scaling Tensor Inference Resolution (Width/Height)')
    ax.set_ylabel('Engine Cycle Output Target (Sec / Single Source)')
    
    # 4. Minority Detection Targeting Impact Analysis
    ax = axes[1, 1]
    scatter = ax.scatter(
        df['conf'], 
        df['minority_recall'], 
        c=df['map50_95'], 
        cmap='magma', 
        s=90, 
        edgecolor='gray', 
        alpha=0.9
    )
    ax.set_title('Minority Sector Representation Detection Index Targets')
    ax.set_xlabel('Probability Rejection Threshold')
    ax.set_ylabel('Recall Distribution Density')
    plt.colorbar(scatter, ax=ax, label='Global Evaluation AP50-95 Impact')
    
    plt.tight_layout()
    plt.savefig(cfg.PLOT_OUTPUT, dpi=300)
    plt.show()
    print(f'Pipeline generated validation insights serialized sequentially into ({cfg.PLOT_OUTPUT})')

## Execution Trigger
Commence the procedural hyperparameter optimization sequence utilizing all designated variables synchronously.

In [ ]:
if __name__ == '__main__':
    print('Initializing automated procedural parameter exploration sequence.')
    
    df_metrics = run_grid_search(config)
    
    if not df_metrics.empty:
        best_config_f1 = df_metrics.sort_values(by='f1_score', ascending=False).iloc[0]
        print('\nOptimal Phase Configuration Metrics Reached:\n')
        print(best_config_f1.to_frame().T)
        
        plot_grid_search_insights(df_metrics, config)
    else:
        print('[WARN] Procedural tracking failed building matrix mappings - Validate model tensor constraints.')